In [0]:
from pyspark.sql import functions as F, Window

CAT = "workspace"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CAT}.gold")

def salvar(df, nome):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{CAT}.gold.{nome}"))

In [0]:
# chaves substitutas geradas com row_number ordenado pela chave natural, assim são as mesmas a cada execução do job. a chave natural fica na dim_movies

# metadados de cada filme
info = spark.table(f"{CAT}.silver.tb_info_filmes")
dim_movies = (info
    .withColumn("sk_movie_id", F.row_number().over(Window.orderBy("id_filme")).cast("bigint"))
    .select("sk_movie_id", "id_filme", "titulo", "data_lancamento", "ano_lancamento",
            "duracao_minutos", "idioma_original", "status_filme", "sinopse"))
salvar(dim_movies, "dim_movies")

# catálogo único de gêneros
gen = spark.table(f"{CAT}.silver.tb_generos").select(F.col("genero").alias("nome_genero")).distinct()
dim_genres = gen.withColumn("sk_genre_id", F.row_number().over(Window.orderBy("nome_genero")).cast("bigint")) \
                .select("sk_genre_id", "nome_genero")
salvar(dim_genres, "dim_genres")

# só pessoas. a mesma pessoa em papéis diferentes vira uma linha por papel
pe = spark.table(f"{CAT}.silver.tb_pessoas_empresas")
pessoas = (pe.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
             .select(F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa")).distinct())
dim_people = pessoas.withColumn("sk_person_id", F.row_number().over(Window.orderBy("tipo_pessoa", "nome_pessoa")).cast("bigint")) \
                    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
salvar(dim_people, "dim_people")

# catálogo único de produtoras
emp = (pe.filter(F.col("tipo_entidade") == "Produtora")
         .select(F.col("nome_entidade").alias("nome_produtora")).distinct())
dim_companies = emp.withColumn("sk_company_id", F.row_number().over(Window.orderBy("nome_produtora")).cast("bigint")) \
                   .select("sk_company_id", "nome_produtora")
salvar(dim_companies, "dim_companies")

# avaliações resumidas por filme
dm = spark.table(f"{CAT}.gold.dim_movies").select("sk_movie_id", "id_filme")
rev = (spark.table(f"{CAT}.silver.tb_avaliacoes_usuarios")
    .groupBy("id_filme")
    .agg(F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
         F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")))
dim_reviews = (rev.join(dm, "id_filme")
    .withColumn("sk_review_id", F.row_number().over(Window.orderBy("sk_movie_id")).cast("bigint"))
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios"))
salvar(dim_reviews, "dim_reviews")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# 1 linha por filme lançado. financeiro e métricas têm 1 linha por filme na silver, então os joins não duplicam o grão. left join pra não perder filme sem dado financeiro
dm = spark.table(f"{CAT}.gold.dim_movies")
fin = spark.table(f"{CAT}.silver.tb_financeiro_filmes")
met = spark.table(f"{CAT}.silver.tb_metricas_engajamento")

fact = (dm.filter(F.col("status_filme") == "Lançado").select("sk_movie_id", "id_filme")
    .join(fin, "id_filme", "left")
    .join(met, "id_filme", "left")
    .select("sk_movie_id",
            F.col("orcamento_usd").cast("decimal(18,2)"), F.col("receita_usd").cast("decimal(18,2)"),
            F.col("lucro_usd").cast("decimal(18,2)"),
            F.col("orcamento_brl").cast("decimal(18,2)"), F.col("receita_brl").cast("decimal(18,2)"),
            F.col("lucro_brl").cast("decimal(18,2)"),
            F.col("popularidade").cast("double"), F.col("nota_media_tmdb").cast("double"),
            F.col("qtd_votos_tmdb").cast("int"), F.col("nota_media_imdb").cast("double"),
            F.col("qtd_votos_imdb").cast("int")))
salvar(fact, "fact_movies_performance")

In [0]:
# tabelas-ponte ligam filme a gênero/pessoa/produtora (N:N) sem duplicar a fato
dm = spark.table(f"{CAT}.gold.dim_movies").select("sk_movie_id", "id_filme")
pe = spark.table(f"{CAT}.silver.tb_pessoas_empresas")

dg = spark.table(f"{CAT}.gold.dim_genres")
salvar(spark.table(f"{CAT}.silver.tb_generos")
    .join(dm, "id_filme").join(dg, F.col("genero") == dg["nome_genero"])
    .select("sk_movie_id", "sk_genre_id").distinct(), "bridge_movie_genre")

dp = spark.table(f"{CAT}.gold.dim_people")
salvar(pe.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(dm, "id_filme")
    .join(dp, (pe["nome_entidade"] == dp["nome_pessoa"]) & (pe["tipo_entidade"] == dp["tipo_pessoa"]))
    .select("sk_movie_id", "sk_person_id").distinct(), "bridge_movie_person")

dc = spark.table(f"{CAT}.gold.dim_companies")
salvar(pe.filter(F.col("tipo_entidade") == "Produtora")
    .join(dm, "id_filme")
    .join(dc, pe["nome_entidade"] == dc["nome_produtora"])
    .select("sk_movie_id", "sk_company_id").distinct(), "bridge_movie_company")

In [0]:
for t in ["dim_movies", "dim_genres", "dim_people", "dim_companies", "dim_reviews",
          "fact_movies_performance", "bridge_movie_genre", "bridge_movie_person", "bridge_movie_company"]:
    print(t, spark.table(f"{CAT}.gold.{t}").count())

f = spark.table(f"{CAT}.gold.fact_movies_performance")
print("fato: linhas", f.count(), "| filmes únicos", f.select("sk_movie_id").distinct().count())
f.printSchema()

dim_movies 97879
dim_genres 19
dim_people 419240
dim_companies 45753
dim_reviews 27303
fact_movies_performance 96522
bridge_movie_genre 140728
bridge_movie_person 763308
bridge_movie_company 119209
fato: linhas 96522 | filmes únicos 96522
root
 |-- sk_movie_id: long (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- popularidade: double (nullable = true)
 |-- nota_media_tmdb: double (nullable = true)
 |-- qtd_votos_tmdb: integer (nullable = true)
 |-- nota_media_imdb: double (nullable = true)
 |-- qtd_votos_imdb: integer (nullable = true)

